# M5 — Suggested route

Suggests the team (`assigned_team`) from the **summary only**, on a **chronological split**. Channel / plan / tenure are excluded (their link to the team drifts month to month). `operations` is merged into `support`. Suggestions below the review threshold (0.5) are marked *needs human review*.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import numpy as np, pandas as pd
import case_pipeline as P   # shared loading / splits / features / evaluation

In [ ]:
df       = P.load_clean(merge_team_aliases=True)   # operations -> support
FEATURES = ["case_summary"]
TARGET   = "assigned_team"
classes  = sorted(df[TARGET].unique())
REVIEW_THRESHOLD = 0.5

train, _, test = P.time_split(df)
model = P.make_clf([P.t_word(), P.t_char()]).fit(train[FEATURES], train[TARGET])
proba = model.predict_proba(test[FEATURES])
pred  = model.classes_[proba.argmax(axis=1)]
confidence = proba.max(axis=1)

baseline = df[TARGET].value_counts(normalize=True).iloc[0]
metrics  = P.evaluate(test[TARGET], pred, classes, baseline=baseline)
metrics["pct_needs_review"] = round(float((confidence < REVIEW_THRESHOLD).mean()), 3)
confident = confidence >= REVIEW_THRESHOLD
metrics["acc_when_confident"] = round(float(P.accuracy_score(test[TARGET][confident], pred[confident])), 3)
metrics

In [ ]:
heldout = test[["case_id", "case_summary", TARGET]].copy().reset_index(drop=True)
heldout["predicted"]  = pred
heldout["confidence"] = confidence.round(3)

deployed = P.make_clf([P.t_word(), P.t_char()]).fit(df[FEATURES], df[TARGET])
P.save_bundle("m5_route", {
    "model": deployed, "target": TARGET, "features": FEATURES, "classes": classes, "kind": "multiclass",
    "label": "Suggested route", "review_threshold": REVIEW_THRESHOLD, "metrics": metrics, "test": heldout,
    "report": P.class_report(test[TARGET], pred, classes),
    "confusion": P.conf_matrix(test[TARGET], pred, classes),
    "meta": {"contributor": "F", "approach": "summary-only router with review gate",
             "split": "time", "inputs": "summary only",
             "note": "operations merged into support; review gate at 0.5"}})